# Train the traffic classifier
YOLO classification `.pt`, with the same stretch-resize preprocessing as the video demo. Edit the settings below before running. Supply labeled images; a raw video alone is not a training dataset. See `../README.md` for setup and dataset layout.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'traffic_training').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the Train folder or its notebooks subfolder.')
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image


In [ ]:
DATA = ROOT / 'datasets' / 'traffic'
OUTPUT = ROOT / 'outputs' / 'cv'
BASE_MODEL = 'yolo11n-cls.pt'  # Local classification .pt or classification YAML also supported.
IMAGE_SIZE = 224
EPOCHS = 50
BATCH = 16
DEVICE = 'cpu'  # Ultralytics accepts '0' for the first CUDA GPU.
SEED = 42
PATIENCE = 12


## Audit labels and splits
Use the same class folders in train/val/test. Split videos by recording/camera/time before extracting frames. Exact duplicate checks do not detect similar neighboring frames. The test images are audited but never used for training or checkpoint selection.

In [ ]:
from traffic_training.classifier import audit_dataset, train_classifier
audit = audit_dataset(DATA)
display(audit)


## Train and export
This cell performs training. A named pretrained checkpoint may download on first use. Output folders with an existing exported model are rejected; select a new output folder for a new experiment.

In [ ]:
metadata = train_classifier(DATA, OUTPUT, model=BASE_MODEL, image_size=IMAGE_SIZE,
    epochs=EPOCHS, batch=BATCH, seed=SEED, device=DEVICE, patience=PATIENCE)
display(metadata)


In [ ]:
results_plot = Path(metadata['training_run']) / 'results.png'
if results_plot.exists(): display(Image(filename=str(results_plot)))
print('Demo model:', OUTPUT / 'classifier.pt')
print('Set the demo ROI resolution to:', IMAGE_SIZE, 'x', IMAGE_SIZE)


Use notebook 02 for held-out evaluation. Load `classifier.pt` in the web/desktop demo or place it in the server model library for Android. The selected video ROI must represent the same content and labels as the training images. The export is a standard Ultralytics checkpoint; training helpers are not needed for inference.